# NusaSiaga Gemma Pipeline

Reproducible Kaggle-style pipeline for transforming wildfire hotspot observations into dashboard-ready environmental disaster intelligence.

This notebook is deterministic, uses inline sample data when no CSV is available, and avoids external API calls. It is prepared for future verified data integration, including NASA FIRMS exports and AQI/OpenAQ enrichment.

## Workflow

1. Load NASA FIRMS-style hotspot data from `data/firms_hotspots.csv` when available.
2. Fall back to inline sample data for reproducible local and Kaggle runs.
3. Clean and normalize hotspot fields.
4. Classify wildfire risk using brightness, confidence, FRP, and distance to settlements when available.
5. Generate structured JSON outputs for the NusaSiaga dashboard.
6. Prepare Gemma prompt examples for environmental disaster analysis.
7. Reserve extension points for future AQI/OpenAQ integration.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 50)

DATA_PATH = Path("../data/firms_hotspots.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

## Input Data

NASA FIRMS exports commonly include latitude, longitude, brightness, confidence, FRP, acquisition date/time, satellite, and instrument fields. This MVP sample also includes Indonesian region labels and approximate distance to settlements so risk classification can account for community exposure.

In [ ]:
sample_hotspots = [
    {
        "latitude": 0.2933,
        "longitude": 101.7068,
        "brightness": 372.4,
        "confidence": 92,
        "frp": 58.2,
        "acq_date": "2026-05-13",
        "acq_time": "0815",
        "satellite": "Aqua",
        "instrument": "MODIS",
        "region": "Riau",
        "distance_to_settlement_km": 2.8,
    },
    {
        "latitude": -0.1322,
        "longitude": 111.0969,
        "brightness": 348.1,
        "confidence": 84,
        "frp": 39.5,
        "acq_date": "2026-05-13",
        "acq_time": "0830",
        "satellite": "Terra",
        "instrument": "MODIS",
        "region": "Kalimantan Barat",
        "distance_to_settlement_km": 6.4,
    },
    {
        "latitude": -3.3194,
        "longitude": 103.9144,
        "brightness": 326.7,
        "confidence": 68,
        "frp": 18.9,
        "acq_date": "2026-05-13",
        "acq_time": "0900",
        "satellite": "Suomi NPP",
        "instrument": "VIIRS",
        "region": "Sumatera Selatan",
        "distance_to_settlement_km": 11.2,
    },
    {
        "latitude": 0.4819,
        "longitude": 102.1265,
        "brightness": 335.2,
        "confidence": 73,
        "frp": 24.1,
        "acq_date": "2026-05-13",
        "acq_time": "0915",
        "satellite": "NOAA-20",
        "instrument": "VIIRS",
        "region": "Riau",
        "distance_to_settlement_km": 4.1,
    },
]

if DATA_PATH.exists():
    raw_df = pd.read_csv(DATA_PATH)
    data_source = str(DATA_PATH)
else:
    raw_df = pd.DataFrame(sample_hotspots)
    data_source = "inline-sample"

raw_df

## Clean and Normalize

The cleaning step standardizes expected columns, converts numeric fields, normalizes timestamps, and fills optional fields with safe defaults. The output should remain stable across local, CI, and Kaggle-style execution.

In [ ]:
REQUIRED_COLUMNS = [
    "latitude",
    "longitude",
    "brightness",
    "confidence",
    "frp",
    "acq_date",
    "acq_time",
    "region",
]

OPTIONAL_DEFAULTS = {
    "satellite": "unknown",
    "instrument": "unknown",
    "distance_to_settlement_km": None,
}

def clean_hotspots(df: pd.DataFrame) -> pd.DataFrame:
    missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    clean = df.copy()
    for column, default in OPTIONAL_DEFAULTS.items():
        if column not in clean.columns:
            clean[column] = default

    numeric_columns = [
        "latitude",
        "longitude",
        "brightness",
        "confidence",
        "frp",
        "distance_to_settlement_km",
    ]
    for column in numeric_columns:
        clean[column] = pd.to_numeric(clean[column], errors="coerce")

    clean["acq_time"] = clean["acq_time"].astype(str).str.zfill(4)
    clean["timestamp_utc"] = pd.to_datetime(
        clean["acq_date"].astype(str) + " " + clean["acq_time"],
        format="%Y-%m-%d %H%M",
        errors="coerce",
        utc=True,
    )
    clean["region"] = clean["region"].fillna("Unknown").astype(str).str.strip()
    clean["satellite"] = clean["satellite"].fillna("unknown").astype(str)
    clean["instrument"] = clean["instrument"].fillna("unknown").astype(str)

    clean = clean.dropna(subset=["latitude", "longitude", "brightness", "confidence", "frp"])
    clean = clean.sort_values(["region", "timestamp_utc", "brightness"], ascending=[True, True, False])
    clean = clean.reset_index(drop=True)
    return clean

hotspots_df = clean_hotspots(raw_df)
hotspots_df

## Deterministic Wildfire Risk Classification

Risk is classified using transparent scoring rules:

- Brightness indicates fire intensity.
- Confidence indicates detection reliability.
- FRP estimates fire radiative power.
- Distance to settlement increases priority when communities are nearby.

These rules are intentionally simple for the MVP and can later be replaced by calibrated thresholds or verified agency guidance.

In [ ]:
def score_hotspot(row: pd.Series) -> int:
    score = 0

    if row["brightness"] >= 360:
        score += 3
    elif row["brightness"] >= 340:
        score += 2
    elif row["brightness"] >= 320:
        score += 1

    if row["confidence"] >= 90:
        score += 3
    elif row["confidence"] >= 75:
        score += 2
    elif row["confidence"] >= 50:
        score += 1

    if row["frp"] >= 50:
        score += 3
    elif row["frp"] >= 25:
        score += 2
    elif row["frp"] >= 10:
        score += 1

    distance = row.get("distance_to_settlement_km")
    if pd.notna(distance):
        if distance <= 3:
            score += 3
        elif distance <= 7:
            score += 2
        elif distance <= 12:
            score += 1

    return score

def classify_score(score: int) -> str:
    if score >= 10:
        return "CRITICAL"
    if score >= 7:
        return "HIGH"
    if score >= 4:
        return "MEDIUM"
    return "LOW"

classified_df = hotspots_df.copy()
classified_df["risk_score"] = classified_df.apply(score_hotspot, axis=1)
classified_df["risk_level"] = classified_df["risk_score"].map(classify_score)
classified_df[["region", "brightness", "confidence", "frp", "distance_to_settlement_km", "risk_score", "risk_level"]]

## Dashboard JSON Output

The following payloads are shaped for future use by NusaSiaga dashboard components: map markers, report cards, and incident feed entries. Files are written to `outputs/` for reproducible artifacts.

In [ ]:
def make_dashboard_payload(df: pd.DataFrame) -> dict[str, Any]:
    markers = []
    reports = []

    for row in df.to_dict(orient="records"):
        timestamp = row["timestamp_utc"].isoformat() if pd.notna(row["timestamp_utc"]) else None
        marker = {
            "area": row["region"],
            "latitude": round(float(row["latitude"]), 5),
            "longitude": round(float(row["longitude"]), 5),
            "risk": row["risk_level"],
            "riskScore": int(row["risk_score"]),
            "brightness": round(float(row["brightness"]), 1),
            "confidence": int(row["confidence"]),
            "frp": round(float(row["frp"]), 1),
            "timestampUtc": timestamp,
            "source": data_source,
        }
        markers.append(marker)
        reports.append(
            {
                "area": row["region"],
                "risk": row["risk_level"],
                "status": f"{row['risk_level']} wildfire hotspot detected",
                "aqiEstimate": "Pending AQI/OpenAQ integration",
                "carbonReleaseEstimate": f"{round(float(row['frp']) * 0.21, 1)} kt indicative",
                "timestampUtc": timestamp,
            }
        )

    return {
        "schemaVersion": "nusasiaga.dashboard.v1",
        "generatedBy": "notebooks/nusasiaga_gemma_pipeline.ipynb",
        "dataSource": data_source,
        "markers": markers,
        "reports": reports,
    }

dashboard_payload = make_dashboard_payload(classified_df)

dashboard_output_path = OUTPUT_DIR / "dashboard_hotspots.json"
dashboard_output_path.write_text(json.dumps(dashboard_payload, indent=2), encoding="utf-8")

dashboard_payload

## Gemma Reasoning Prompt Examples

These prompts are examples for local Gemma/Ollama analysis. They are not executed in this notebook, keeping the workflow offline and deterministic.

In [ ]:
def build_gemma_prompt(marker: dict[str, Any]) -> str:
    return f"""
You are NusaSiaga, an offline-first environmental disaster intelligence assistant.
Analyze this wildfire hotspot and return only valid JSON with:
- severity
- aqiEstimate
- evacuationRecommendation
- environmentalImpact

Hotspot:
- Region: {marker['area']}
- Risk: {marker['risk']}
- Risk score: {marker['riskScore']}
- Brightness: {marker['brightness']}
- Confidence: {marker['confidence']}
- FRP: {marker['frp']}
- Timestamp UTC: {marker['timestampUtc']}
""".strip()

gemma_prompt_examples = [build_gemma_prompt(marker) for marker in dashboard_payload["markers"]]

prompt_output_path = OUTPUT_DIR / "gemma_prompt_examples.json"
prompt_output_path.write_text(json.dumps(gemma_prompt_examples, indent=2), encoding="utf-8")

print(gemma_prompt_examples[0])

## Future AQI/OpenAQ Integration

The MVP does not call external APIs. Future integration can enrich each hotspot with nearby AQI observations, PM2.5, PM10, and station metadata. Keep this enrichment as a separate deterministic step that can cache raw responses and preserve source metadata.

In [ ]:
def attach_aqi_placeholder(payload: dict[str, Any]) -> dict[str, Any]:
    enriched = json.loads(json.dumps(payload))
    enriched["aqiIntegration"] = {
        "status": "not-connected",
        "plannedProvider": "OpenAQ",
        "notes": "Future step: match hotspot coordinates to nearest AQI station or gridded air quality estimate.",
    }
    return enriched

future_ready_payload = attach_aqi_placeholder(dashboard_payload)
future_ready_payload["aqiIntegration"]

## Reproducibility Notes

- No external API calls are required.
- Inline data guarantees the notebook can run before verified CSV data is added.
- Classification thresholds are transparent and deterministic.
- Outputs are JSON artifacts ready for dashboard integration experiments.